In [ ]:
%pip install -r requirements.txt

In [ ]:
import pandas as pd
import numpy as np
import random
import gc
import os
import time
import dask.array as da
from joblib import Parallel, delayed
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
from scipy.stats import gamma  
from IPython.display import display
from dask import delayed

start_time = time.time()

In [ ]:
# Credit Risk Parameters (See Gordy Model or section 3 of my thesis: Transition Risk Add-on: a simulation approach. pdf for details)
mean = 1
variance = 4
k = mean ** 2 / variance
theta = variance / mean
E_LGD = 0.50
sigma = 2
lambda_ = 0.5
eta = 0.25
sigma2 = sigma ** 2

#Climate Risk Parameters
model = 'WITCH' #Possible Choices: REMIND, WITCH
scenario = 'LIMITS450' #Possible Choices: LIMITSOilIndependence, LIMITS450, LIMITS500, LIMITSPledges, LIMITSEnergyIndependence
Selected_Region = 'Europe' #Possible Choices: Asia, Europe, LAM, NAM, ROW, Africa, World
num_simulations = 10000 #Number of simulations to run / Number of Monte Carlo iterations / Number of random portfolios to generate
max_subsectors_per_sector = 3 #Max number of subsectors to consider for each sector
sectors_data = {
    'primary_fossil': ['Primary Energy|Coal', 'Primary Energy|Gas', 'Primary Energy|Oil'],
    'secondary_fossil': ['Secondary Energy|Electricity|Coal', 'Secondary Energy|Electricity|Gas', 'Secondary Energy|Electricity|Oil'],
    'secondary_renewable': ['Secondary Energy|Electricity|Biomass', 'Secondary Energy|Electricity|Geothermal',
                            'Secondary Energy|Electricity|Solar|CSP', 'Secondary Energy|Electricity|Nuclear',
                            'Secondary Energy|Electricity|Solar|PV', 'Secondary Energy|Electricity|Wind', 'Secondary Energy|Electricity|Hydro']
}
max_percentages = { #Max exposure for each energy category
    'primary_fossil': 50,
    'secondary_fossil': 50,
    'secondary_renewable': 0
}
risk_free_rate = 0.03 #Risk-free rate for discounting climate losses
CONFIDENCE_LEVEL = 0.995


In [ ]:
folder = 'data'
file_portfolio = f"portafoglio_{Selected_Region}.csv"
file_shocks = f"shock_{Selected_Region}.csv"
portfolio = pd.read_csv(os.path.join(folder, file_portfolio))
shock_data = pd.read_csv(os.path.join(folder, file_shocks))
portfolio.rename(columns={'Regione': 'REGION'}, inplace=True)
shock_data = shock_data[(shock_data['MODEL'] == model) & (shock_data['SCENARIO'] == scenario)]
shock_data = shock_data.round(3)
print (portfolio.head())
print (shock_data.head())



In [ ]:
#To compute Credit Risk, granularity add-on by Gordy:

SEED_CREDIT = 42
np.random.seed(SEED_CREDIT)

random_values = gamma.rvs(k, scale=theta, size=num_simulations)

percentile_99_5 = np.percentile(random_values, CONFIDENCE_LEVEL * 100)

rating = ['A', 'BBB', 'BB', 'B', 'CCC']
p = np.array([0.06, 0.20, 1.25, 6.25, 17.50])
w = np.array([1.011, 0.836, 0.602, 0.415, 0.295])

rating_table = pd.DataFrame({
    'Rating': rating,
    'p_mean': p,
    'w': w
})

rating_table['mu_x'] = E_LGD * rating_table['p_mean'] * (1 + rating_table['w'] * (percentile_99_5 - 1))

rating_table['Beta'] = (1 / (2 * lambda_)) * (lambda_ ** 2 + eta ** 2) * (
        (1 / sigma2) * (1 + (sigma2 - 1) / percentile_99_5) *
        (percentile_99_5 + (1 - rating_table['w']) / rating_table['w']) - 1)

n_values = [5000, 2000, 1000, 500, 200, 100]
column_names = ['VaR5000', 'VaR2000', 'VaR1000', 'VaR500', 'VaR200', 'VaR100']

for i, n in enumerate(n_values):
    rating_table[column_names[i]] = rating_table['mu_x'] + (rating_table['Beta'] / n) * 100

rating_table = rating_table.drop(columns=['Beta'])

print(rating_table)


In [ ]:
#Build the portfolio for simulating climate exposure: 
np.random.seed(SEED_CREDIT) 
num_bonds = len(portfolio)
portfolio['LGD_j'] = np.random.gamma(shape=lambda_, scale=eta, size=num_bonds)
portfolio = portfolio.round(3)
regions = ['NORTH_AM', 'EUROPE', 'ASIA', 'AFRICA', 'LATIN_AM', 'REST_WORLD']
available_regions = portfolio['REGION'].unique().tolist()
years = ['2025', '2030', '2035', '2040', '2045', '2050']
agg_dict = {year: 'max' for year in years}
max_shock_by_region = shock_data.groupby(['REGION', 'MODEL', 'SCENARIO'], as_index=False).agg(agg_dict)
rename_dict = {year: f'Max_Shock_{year}' for year in years}
max_shock_by_region.rename(columns=rename_dict, inplace=True)

all_allowed_sectors = []
for sector_cat, percentage in max_percentages.items():
    if percentage > 0:
        all_allowed_sectors.extend(sectors_data[sector_cat])

all_sims_list = []

print(f"🎲 Simulations starting ({num_simulations})...")

for sim in tqdm(range(1, num_simulations + 1)):
    temp_df = portfolio[['Titolo', 'REGION', 'LGD_j']].copy()
    temp_df['simulation_num'] = sim
    temp_df['sector'] = [random.sample(all_allowed_sectors, min(3, len(all_allowed_sectors))) 
                         for _ in range(len(temp_df))]
    temp_df = temp_df.explode('sector')
    all_sims_list.append(temp_df)
results_df = pd.concat(all_sims_list).reset_index(drop=True)

print(f"✅ results_df creato con {len(results_df)} righe.")

# Weight generation (see transition shock formula based on GVA)
results_df['weight'] = np.random.randint(1, 100, size=len(results_df))
group_sums = results_df.groupby(['simulation_num', 'Titolo'])['weight'].transform('sum')
results_df['weight'] = np.round((results_df['weight'] / group_sums) * 100).astype(int)
current_sums = results_df.groupby(['simulation_num', 'Titolo'])['weight'].transform('sum')
difference = 100 - current_sums
first_row_mask = ~results_df.duplicated(subset=['simulation_num', 'Titolo'])
results_df.loc[first_row_mask, 'weight'] += difference[first_row_mask]
results_df['weight'] = results_df['weight'].clip(lower=0) #This guarantees that we don't have negative weights

print(f"✅ Weights generated and verified for {len(results_df)} rows.")
print(results_df.head())

In [ ]:
# Transition shocks computation
merged_shocks = results_df.merge(
    shock_data, 
    left_on=['REGION', 'sector'], 
    right_on=['REGION', 'VARIABLE'], 
    how='inner'
)
years_str = [str(y) for y in range(2025, 2055, 5)]

for yr in years_str:
    merged_shocks[f'weighted_shock_{yr}'] = (merged_shocks[yr] * merged_shocks['weight']) / 100
weighted_cols = [f'weighted_shock_{yr}' for yr in years_str]
total_shocks = merged_shocks.groupby(
    ['simulation_num', 'Titolo', 'MODEL', 'SCENARIO', 'REGION', 'LGD_j'], 
    as_index=False
)[weighted_cols].sum()
total_shocks.rename(columns={f'weighted_shock_{yr}': f'shock_tot_{yr}' for yr in years_str}, inplace=True)
max_cols = [f'Max_Shock_{yr}' for yr in years_str]
total_shocks = total_shocks.merge(
    max_shock_by_region[['REGION', 'MODEL', 'SCENARIO'] + max_cols],
    on=['REGION', 'MODEL', 'SCENARIO'],
    how='left'
)
total_shocks.rename(columns={f'Max_Shock_{yr}': f'max_shock_{yr}' for yr in years_str}, inplace=True)

print(f"✅ Shock computed across all issuers.")
display(total_shocks.head())

In [ ]:
years_list = list(range(2025, 2055, 5))

# Climate transition risk losses calculation (Delta V) - Vectorized
for year in years_list:
    s_tot = f'shock_tot_{year}'
    s_max = f'max_shock_{year}'
    dv_col = f'delta_v_{year}'
    total_shocks[dv_col] = - (total_shocks['LGD_j'] * total_shocks[s_tot]) / (total_shocks[s_max] + 1)
delta_v_cols = [f'delta_v_{year}' for year in years_list]

aggregated_losses = total_shocks.groupby(
    ['simulation_num', 'MODEL', 'SCENARIO'], 
    as_index=False
)[delta_v_cols].sum()

rename_dict = {f'delta_v_{year}': f'sum_delta_v_{year}' for year in years_list}
aggregated_losses.rename(columns=rename_dict, inplace=True)

display(aggregated_losses.head())

In [ ]:
# Compute the transition risk add-on by applying a 99.5 percentile VaR to the empirical distribution of climate losses and discounting back according to Battiston model.
years_list = list(range(2025, 2055, 5))
loss_cols = [f'sum_delta_v_{year}' for year in years_list]
percentile_results = aggregated_losses.groupby(['MODEL', 'SCENARIO'])[loss_cols].quantile(CONFIDENCE_LEVEL).reset_index()
rename_percentile = {col: f'percentile_99.5_{year}' for col, year in zip(loss_cols, years_list)}
percentile_results.rename(columns=rename_percentile, inplace=True)

adjusted_table = percentile_results.copy()

for year in years_list:
    T = year - 2025
    discount = np.exp(-risk_free_rate * T)
    
    perc_col = f'percentile_99.5_{year}'
    adj_col = f'adjusted_{year}'
    adjusted_table[adj_col] = adjusted_table[perc_col] * discount
    adjusted_table[adj_col] = adjusted_table[adj_col].apply(lambda x: f"{x:.3f}%")

cols_to_keep = ['MODEL', 'SCENARIO'] + [f'adjusted_{year}' for year in years_list]
adjusted_table = adjusted_table[cols_to_keep]

print(adjusted_table)